In [ ]:
from google.colab import drive
import os
import json
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(PROJECT_PATH, "data", "processed")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "outputs")
MODEL_PATH = os.path.join(PROJECT_PATH, "models")
TABLE_PATH = os.path.join(PROJECT_PATH, "reports", "tables")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["Datetime", "StationId"]
).reset_index(drop=True)

print("Dataset restored successfully.")
print("Shape:", feature_df.shape)
print("Stations:", feature_df["StationId"].nunique())
print("Start:", feature_df["Datetime"].min())
print("End:", feature_df["Datetime"].max())

In [ ]:
current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    col for col in feature_df.columns
    if "_lag_" in col
]

rolling_features = [
    col for col in feature_df.columns
    if "rolling_" in col
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

numeric_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
)

categorical_features = [
    "StationId"
]

target = "AQI_t_plus_1"

all_features = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)
print("Total features:", len(all_features))
print("Target:", target)

In [ ]:
unique_times = np.sort(
    feature_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[split_position - 1]

train_df = feature_df[
    feature_df["Datetime"] <= train_end_time
].copy()

test_df = feature_df[
    feature_df["Datetime"] > train_end_time
].copy()

print("TRAIN")
print(
    train_df["Datetime"].min(),
    "→",
    train_df["Datetime"].max()
)
print("Rows:", len(train_df))

print("\nTEST")
print(
    test_df["Datetime"].min(),
    "→",
    test_df["Datetime"].max()
)
print("Rows:", len(test_df))

In [ ]:
print(
    "Latest train timestamp:",
    train_df["Datetime"].max()
)

print(
    "Earliest test timestamp:",
    test_df["Datetime"].min()
)

print(
    "Leakage-free split:",
    train_df["Datetime"].max()
    < test_df["Datetime"].min()
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

baseline_actual = test_df[target]
baseline_pred = test_df["AQI"]

baseline_mae = mean_absolute_error(
    baseline_actual,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_actual,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    baseline_actual,
    baseline_pred
)

print("Persistence Baseline")
print("--------------------")
print("MAE :", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))
print("R²  :", round(baseline_r2, 4))

In [ ]:
X_train = train_df[all_features]
y_train = train_df[target]

X_test = test_df[all_features]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Leakage-safe preprocessing pipeline created.")

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(
    X_train,
    y_train
)

linear_pred = linear_model.predict(
    X_test
)

linear_mae = mean_absolute_error(
    y_test,
    linear_pred
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_pred
    )
)

linear_r2 = r2_score(
    y_test,
    linear_pred
)

print("Linear Regression")
print("-----------------")
print("MAE :", round(linear_mae, 4))
print("RMSE:", round(linear_rmse, 4))
print("R²  :", round(linear_r2, 4))

In [ ]:
print(
    "MAE improvement (%):",
    round(
        ((baseline_mae - linear_mae) / baseline_mae) * 100,
        2
    )
)

print(
    "RMSE improvement (%):",
    round(
        ((baseline_rmse - linear_rmse) / baseline_rmse) * 100,
        2
    )
)

In [ ]:
train_times = np.sort(
    train_df["Datetime"].unique()
)

n_folds = 4

fold_time_groups = np.array_split(
    train_times,
    n_folds + 1
)

validation_folds = []

for fold_number in range(1, len(fold_time_groups)):

    fold_train_times = np.concatenate(
        fold_time_groups[:fold_number]
    )

    fold_validation_times = fold_time_groups[
        fold_number
    ]

    validation_folds.append(
        {
            "Fold": fold_number,
            "TrainEnd": fold_train_times.max(),
            "ValidationStart": fold_validation_times.min(),
            "ValidationEnd": fold_validation_times.max()
        }
    )

fold_table = pd.DataFrame(
    validation_folds
)

display(fold_table)

In [ ]:
fold_table["LeakageFree"] = (
    fold_table["TrainEnd"]
    < fold_table["ValidationStart"]
)

display(fold_table)

print(
    "All folds leakage-free:",
    fold_table["LeakageFree"].all()
)

In [ ]:
from sklearn.base import clone

rolling_results = []

for fold in validation_folds:

    fold_train_mask = (
        train_df["Datetime"]
        <= fold["TrainEnd"]
    )

    fold_val_mask = (
        (train_df["Datetime"] >= fold["ValidationStart"])
        &
        (train_df["Datetime"] <= fold["ValidationEnd"])
    )

    X_fold_train = train_df.loc[
        fold_train_mask,
        all_features
    ]

    y_fold_train = train_df.loc[
        fold_train_mask,
        target
    ]

    X_fold_val = train_df.loc[
        fold_val_mask,
        all_features
    ]

    y_fold_val = train_df.loc[
        fold_val_mask,
        target
    ]

    fold_model = clone(
        linear_model
    )

    fold_model.fit(
        X_fold_train,
        y_fold_train
    )

    fold_pred = fold_model.predict(
        X_fold_val
    )

    rolling_results.append(
        {
            "Fold": fold["Fold"],
            "MAE": mean_absolute_error(
                y_fold_val,
                fold_pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_fold_val,
                    fold_pred
                )
            ),
            "R2": r2_score(
                y_fold_val,
                fold_pred
            )
        }
    )

rolling_linear_results = pd.DataFrame(
    rolling_results
)

display(rolling_linear_results)

In [ ]:
print("Rolling Linear Regression")
print("-------------------------")

print(
    "Mean MAE :",
    round(
        rolling_linear_results["MAE"].mean(),
        4
    )
)

print(
    "Mean RMSE:",
    round(
        rolling_linear_results["RMSE"].mean(),
        4
    )
)

print(
    "Mean R²  :",
    round(
        rolling_linear_results["R2"].mean(),
        4
    )
)

print(
    "MAE Std  :",
    round(
        rolling_linear_results["MAE"].std(),
        4
    )
)

In [ ]:
!pip install -q xgboost

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    )
                ]
            ),
            categorical_features
        )
    ]
)

print("Tree-model preprocessing pipeline created.")

In [ ]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_model = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=12,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

decision_tree_model.fit(
    X_train,
    y_train
)

tree_pred = decision_tree_model.predict(
    X_test
)

tree_mae = mean_absolute_error(
    y_test,
    tree_pred
)

tree_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tree_pred
    )
)

tree_r2 = r2_score(
    y_test,
    tree_pred
)

print("Decision Tree")
print("-------------")
print("MAE :", round(tree_mae, 4))
print("RMSE:", round(tree_rmse, 4))
print("R²  :", round(tree_r2, 4))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=150,
                max_depth=18,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)

rf_pred = random_forest_model.predict(
    X_test
)

rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_pred
    )
)

rf_r2 = r2_score(
    y_test,
    rf_pred
)

print("Random Forest")
print("-------------")
print("MAE :", round(rf_mae, 4))
print("RMSE:", round(rf_rmse, 4))
print("R²  :", round(rf_r2, 4))

In [ ]:
from xgboost import XGBRegressor

xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                eval_metric="rmse",
                random_state=42,
                n_jobs=-1,
                tree_method="hist"
            )
        )
    ]
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_pred = xgb_model.predict(
    X_test
)

xgb_mae = mean_absolute_error(
    y_test,
    xgb_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_pred
    )
)

xgb_r2 = r2_score(
    y_test,
    xgb_pred
)

print("XGBoost")
print("-------")
print("MAE :", round(xgb_mae, 4))
print("RMSE:", round(xgb_rmse, 4))
print("R²  :", round(xgb_r2, 4))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        tree_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        tree_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R2": [
        baseline_r2,
        linear_r2,
        tree_r2,
        rf_r2,
        xgb_r2
    ]
})

results = results.sort_values(
    "RMSE"
).reset_index(drop=True)

display(results)

In [ ]:
results["MAE_Improvement_vs_Baseline_%"] = (
    (
        baseline_mae - results["MAE"]
    )
    / baseline_mae
) * 100

results["RMSE_Improvement_vs_Baseline_%"] = (
    (
        baseline_rmse - results["RMSE"]
    )
    / baseline_rmse
) * 100

display(results)

In [ ]:
results_path = os.path.join(
    TABLE_PATH,
    "phase6_classical_model_results.csv"
)

results.to_csv(
    results_path,
    index=False
)

rolling_results_path = os.path.join(
    TABLE_PATH,
    "phase6_rolling_linear_results.csv"
)

rolling_linear_results.to_csv(
    rolling_results_path,
    index=False
)

print("Results saved.")
print(results_path)
print(rolling_results_path)

In [ ]:
import joblib

joblib.dump(
    linear_model,
    os.path.join(
        MODEL_PATH,
        "phase6_linear_regression.joblib"
    )
)

joblib.dump(
    decision_tree_model,
    os.path.join(
        MODEL_PATH,
        "phase6_decision_tree.joblib"
    )
)

joblib.dump(
    random_forest_model,
    os.path.join(
        MODEL_PATH,
        "phase6_random_forest.joblib"
    )
)

joblib.dump(
    xgb_model,
    os.path.join(
        MODEL_PATH,
        "phase6_xgboost.joblib"
    )
)

print("All Phase 6 classical models saved.")

In [ ]:
phase6_summary = {
    "completed_phase": "Phase 6 - Classical Baselines and Model Training",
    "next_phase": "Phase 7 - Model Evaluation and Ablation Study",

    "train_start": str(train_df["Datetime"].min()),
    "train_end": str(train_df["Datetime"].max()),
    "test_start": str(test_df["Datetime"].min()),
    "test_end": str(test_df["Datetime"].max()),

    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),

    "baseline": {
        "MAE": float(baseline_mae),
        "RMSE": float(baseline_rmse),
        "R2": float(baseline_r2)
    },

    "linear_regression": {
        "MAE": float(linear_mae),
        "RMSE": float(linear_rmse),
        "R2": float(linear_r2)
    },

    "decision_tree": {
        "MAE": float(tree_mae),
        "RMSE": float(tree_rmse),
        "R2": float(tree_r2)
    },

    "random_forest": {
        "MAE": float(rf_mae),
        "RMSE": float(rf_rmse),
        "R2": float(rf_r2)
    },

    "xgboost": {
        "MAE": float(xgb_mae),
        "RMSE": float(xgb_rmse),
        "R2": float(xgb_r2)
    }
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase6_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        phase6_summary,
        f,
        indent=4
    )

print("Phase 6 summary saved:")
print(summary_path)

In [ ]:
print("=" * 60)
print("PHASE 6 — CLASSICAL MODEL TRAINING COMPLETE")
print("=" * 60)

display(results)

print("\nRolling Linear Regression:")
display(rolling_linear_results)

print("=" * 60)

In [ ]:
import os
import json
import joblib
import pandas as pd

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH, "models"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)


# --------------------------------------------------
# 1. Save model predictions
# --------------------------------------------------

prediction_df = test_df[
    [
        "StationId",
        "StationName",
        "Datetime",
        "AQI",
        "AQI_t_plus_1"
    ]
].copy()

prediction_df["Persistence_Prediction"] = baseline_pred.values
prediction_df["LinearRegression_Prediction"] = linear_pred
prediction_df["DecisionTree_Prediction"] = tree_pred
prediction_df["RandomForest_Prediction"] = rf_pred
prediction_df["XGBoost_Prediction"] = xgb_pred

prediction_path = os.path.join(
    OUTPUT_PATH,
    "phase6_test_predictions.parquet"
)

prediction_df.to_parquet(
    prediction_path,
    index=False
)


# --------------------------------------------------
# 2. Save model comparison
# --------------------------------------------------

results_path = os.path.join(
    TABLE_PATH,
    "phase6_classical_model_results.csv"
)

results.to_csv(
    results_path,
    index=False
)


# --------------------------------------------------
# 3. Save rolling validation results
# --------------------------------------------------

rolling_path = os.path.join(
    TABLE_PATH,
    "phase6_rolling_linear_results.csv"
)

rolling_linear_results.to_csv(
    rolling_path,
    index=False
)


# --------------------------------------------------
# 4. Save fitted models
# --------------------------------------------------

model_files = {
    "linear_regression": linear_model,
    "decision_tree": decision_tree_model,
    "random_forest": random_forest_model,
    "xgboost": xgb_model
}

for model_name, model in model_files.items():

    model_path = os.path.join(
        MODEL_PATH,
        f"phase6_{model_name}.joblib"
    )

    joblib.dump(
        model,
        model_path
    )


# --------------------------------------------------
# 5. Save Phase 6 summary
# --------------------------------------------------

phase6_summary = {
    "completed_phase": "Phase 6 - Classical Baselines and Model Training",
    "next_phase": "Phase 7 - Model Evaluation and Ablation Study",

    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),

    "train_start": str(train_df["Datetime"].min()),
    "train_end": str(train_df["Datetime"].max()),

    "test_start": str(test_df["Datetime"].min()),
    "test_end": str(test_df["Datetime"].max()),

    "models": {
        row["Model"]: {
            "MAE": float(row["MAE"]),
            "RMSE": float(row["RMSE"]),
            "R2": float(row["R2"])
        }
        for _, row in results.iterrows()
    },

    "feature_counts": {
        "current": len(current_features),
        "lag": len(lag_features),
        "rolling": len(rolling_features),
        "calendar": len(calendar_features),
        "events": len(event_features),
        "spatial": len(categorical_features)
    }
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase6_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        phase6_summary,
        f,
        indent=4
    )


# --------------------------------------------------
# 6. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 6 - Classical Baselines and Model Training",
    "next_phase": "Phase 7 - Model Evaluation and Ablation Study",
    "prediction_target": "AQI_t_plus_1",
    "test_predictions": "phase6_test_predictions.parquet"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(status_path, "w") as f:
    json.dump(
        session_status,
        f,
        indent=4
    )


# --------------------------------------------------
# 7. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    prediction_path,
    results_path,
    rolling_path,
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 6 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nSaved models:")
for model_name in model_files:
    print(
        "OK   ",
        os.path.join(
            MODEL_PATH,
            f"phase6_{model_name}.joblib"
        )
    )

print("\nCompleted: Phase 6")
print("Next: Phase 7 - Model Evaluation and Ablation Study")